# 03 · Build the feature matrix

**Question:** which columns are legitimately available at request time, and how are they encoded without leaking across the split?

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

# Anchor on this project specifically: it sits in a subdirectory of a repository
# that has its own pyproject.toml, so "nearest pyproject.toml" is not enough.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "transfer_decline").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))


## Leakage: what has to stay out
Anything recorded after the transfer-center decision, or that encodes the decision itself, cannot be a feature. Using it would let the model 'predict' the decision from its own consequences.

In [2]:
from transfer_decline.data import POST_DECISION_COLUMNS
from transfer_decline.features import FEATURE_COLUMNS, FORBIDDEN, NUMERIC_FEATURES, CATEGORICAL_FEATURES
print('excluded as post-decision / leakage:')
for c in POST_DECISION_COLUMNS: print('  -', c)
print()
print('payer is also excluded from the model — audited separately in notebook 04')
assert not (set(FEATURE_COLUMNS) & FORBIDDEN)

excluded as post-decision / leakage:
  - decision_datetime
  - bed_assignment_datetime
  - arrival_datetime
  - disposition
  - decline_reason
  - inpatient_los_days
  - icu_upgrade_within_24h

payer is also excluded from the model — audited separately in notebook 04


In [3]:
df = pd.read_csv(ROOT / 'data' / 'synthetic' / 'prepared_transfer_requests.csv')
print(f'{len(FEATURE_COLUMNS)} features')
print('numeric:', NUMERIC_FEATURES)
print('categorical:', CATEGORICAL_FEATURES)
from transfer_decline.features import build_feature_frame
X = build_feature_frame(df)
display(X.head())

17 features
numeric: ['patient_age', 'distance_miles', 'acuity_score', 'icu_occupancy_pct', 'medsurg_occupancy_pct', 'ed_boarding_count', 'request_hour', 'request_dow', 'request_month']
categorical: ['referring_facility', 'referring_facility_type', 'transport_mode', 'requested_service_line', 'requested_level_of_care', 'sex']


,patient_age,distance_miles,acuity_score,icu_occupancy_pct,medsurg_occupancy_pct,ed_boarding_count,request_hour,request_dow,request_month,is_weekend,is_overnight,referring_facility,referring_facility_type,transport_mode,requested_service_line,requested_level_of_care,sex
0,79,11.5,2.0,0.977,0.910,9,5,2,1,0,1,Referring Facility L,Community Hospital ED,Ground ALS,Neurology / Stroke,Med-Surg,Male
1,74,39.8,3.0,0.790,1.000,20,9,2,1,0,0,Referring Facility D,Community Hospital ED,Referring Facility Transport,Medical / Hospitalist,Med-Surg,Male
2,67,9.0,3.0,1.000,0.823,4,9,2,1,0,0,Referring Facility D,Community Hospital ED,Ground ALS,Vascular,Med-Surg,Male
3,87,7.7,4.0,0.910,0.956,15,11,2,1,0,0,Referring Facility I,Community Hospital Inpatient,Ground ALS,Orthopedics,Progressive Care / Stepdown,Female
4,56,3.0,4.0,0.832,0.845,9,12,2,1,0,0,Referring Facility E,Critical Access Hospital,Ground ALS,Cardiology,ICU,Male


## Encoding, fit on training only
`make_encoder()` is a `ColumnTransformer`: median imputation + standardisation for numerics (with a missing-indicator column), most-frequent imputation + one-hot for categoricals, rare levels (< 20) folded together. It is `fit` on the training rows and only `transform`-ed on validation and test.

In [4]:
from transfer_decline.features import make_encoder, TARGET
train = df[df['split'] == 'train']
encoder = make_encoder().fit(build_feature_frame(train), train[TARGET])
matrix = encoder.transform(build_feature_frame(df.head()))
print('encoded width:', matrix.shape[1])
print(list(encoder.get_feature_names_out())[:12], '...')

encoded width: 49
['patient_age', 'distance_miles', 'acuity_score', 'icu_occupancy_pct', 'medsurg_occupancy_pct', 'ed_boarding_count', 'request_hour', 'request_dow', 'request_month', 'missingindicator_distance_miles', 'missingindicator_acuity_score', 'is_weekend'] ...


## Note
The encoder object is not saved here — the modelling pipeline in notebook 05 rebuilds and refits it as its first step, so training-only fitting is guaranteed by construction.